# 09 — Attacking-direction diagnostic (goalkeeper cross-check)

**Research status:** Diagnostic inference, **not yet verified** attacking direction. We need a team-by-period sign before interpreting forward/backward progression. Infer each goalkeeper's median x position on exact possession-start snapshots, check agreement between opposing goalkeepers, and flag weak or conflicting cases. **Never silently assign a direction where the evidence is insufficient.**

This consumes the canonical decision snapshots from Notebook 05 and per-match roster metadata. It does not train models, include future outcomes as features, or alter source data. The goalkeeper-based direction inferred here is a *geometric hypothesis* to be corroborated with the documented coordinate convention or a representative match visual inspection before use in the research's primary variables.

**Outputs:** `outputs/P2_attacking_direction_gate.json` and `outputs/P2_attacking_direction_diagnostic.csv`. Upload only the gate JSON.

In [1]:
from pathlib import Path
from collections import defaultdict, Counter
from statistics import median
from datetime import datetime, timezone
import json, csv, math

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'data' / 'matches').is_dir()), None)
if ROOT is None: raise FileNotFoundError('Run from your SkillCorner clone.')
OUT = ROOT / 'outputs'
DATA = ROOT / 'data' / 'matches'
paths = sorted(OUT.glob('P1_canonical_decisions_match_*_ROLES.jsonl'))
assert len(paths) == 20, f'Expected 20 Notebook 05 canonical files, got {len(paths)}'
print('Found',len(paths),'canonical match files')

Found 20 canonical match files


## Extract goalkeeper positional evidence

A keeper is selected only if the metadata's `player_role` explicitly identifies a goalkeeper. All available keeper coordinates are taken from the *same exact-frame decision snapshots* as the other geometry. We use *detected positions only* for primary directional evidence; extrapolated goalkeeper samples are counted but not substituted. One team's goalkeeper can be the carrier or may appear as a teammate or opponent of the event carrier.


In [2]:
def valid_xy(pt):
    return (isinstance(pt,list) and len(pt)==2 and
            all(isinstance(v,(int,float)) and math.isfinite(v) for v in pt))

def keeper_ids(metadata):
    ids = defaultdict(set)
    for player in metadata.get('players', []):
        role=player.get('player_role') or {}
        terms=' '.join(str(role.get(k) or '') for k in ('name','acronym','position_group')).lower()
        if 'goalkeeper' in terms or terms.strip() in ('gk','goalie'):
            ids[str(player['team_id'])].add(str(player['id']))
    return ids

def samples_for_match(path):
    mid=path.name.split('_match_')[1].split('_ROLES')[0]
    meta=json.loads((DATA/mid/f'{mid}_match.json').read_text())
    team_ids={str(meta['home_team']['id']),str(meta['away_team']['id'])}
    ids=keeper_ids(meta)
    if set(ids)!=team_ids:
        raise AssertionError(f'{mid}: goalkeeper metadata incomplete: {dict(ids)}')
    pid_to_team={pid:team for team,players in ids.items() for pid in players}
    obs=defaultdict(list)
    count=Counter()
    with path.open() as f:
        for line in f:
            row=json.loads(line)
            period=row['period']; own=row['attacking_team_id']; carrier=row['carrier_player_id']
            if own not in team_ids:raise AssertionError(f'{mid}: unknown attacking team {own}')
            # canonical player bins already indicate teammate/opponent team roles
            for quality in ('detected','extrapolated'):
                for role in ('teammates','opponents'):
                    team=own if role=='teammates' else next(iter(team_ids-{own}))
                    for p in row.get(f'{role}_{quality}',[]):
                        pid=str(p.get('player_id'))
                        if pid in ids[team] and valid_xy(p.get('xy')):
                            obs[(period,team,quality)].append(float(p['xy'][0]))
            if carrier in pid_to_team and valid_xy(row.get('carrier_xy')):
                team=pid_to_team[carrier]
                if team!=own:raise AssertionError(f'{mid}: keeper carrier team conflict')
                quality='detected' if row.get('carrier_detected') is True else 'extrapolated'
                obs[(period,team,quality)].append(float(row['carrier_xy'][0]))
            count[period]+=1
    return mid,team_ids,obs,count


## Candidate signs and confidence diagnostics

With pitch-centered x coordinates, a team's goalkeeper normally defends the goal toward the goalkeeper's own end. A proposed attacking sign is therefore opposite the goalkeeper median x sign. This inference **requires confirmation** of the x-axis and goal convention, is invalid if the keeper has advanced unusually, and may fail in non-representative snapshots.

Provisional diagnostic criteria: at least 10 directly detected goalkeeper samples for each team-period; at least 70% of sampled positions on the same side of midfield; each goalkeeper median at least 10 m from x=0; and opposite proposed attacking directions for opponents. These are transparent inspection criteria, *not source-defined facts or statistically validated thresholds*.

In [3]:
diagnostics=[]
for path in paths:
    mid,teams,obs,counts=samples_for_match(path)
    periods=sorted(counts)
    for period in periods:
        pair=[]
        for team in sorted(teams):
            detected=obs.get((period,team,'detected'),[])
            extrap=obs.get((period,team,'extrapolated'),[])
            n=len(detected)
            med=median(detected) if detected else None
            side_fraction=(sum((v>0)==(med>0) for v in detected)/n) if med is not None else None
            eligible=(n>=10 and abs(med)>=10 and side_fraction>=0.70) if med is not None else False
            proposed_sign=(-1 if med>0 else +1) if eligible else None
            entry={'match_id':mid,'period':period,'team_id':team,'decision_snapshots':counts[period],
                   'keeper_samples_detected':n,'keeper_samples_extrapolated':len(extrap),
                   'keeper_x_median_m':med,'keeper_side_agreement':side_fraction,
                   'individual_evidence_eligible':eligible,'proposed_attack_x_sign':proposed_sign}
            pair.append(entry)
        consistent=all(e['individual_evidence_eligible'] for e in pair) and sum(e['proposed_attack_x_sign'] for e in pair)==0
        for e in pair:
            e['opposing_keepers_consistent']=consistent
            e['diagnostic_status']='consistent_provisional' if consistent else 'unresolved'
            diagnostics.append(e)
print('Team-period combinations:',len(diagnostics))
print('Status counts:',dict(Counter(d['diagnostic_status'] for d in diagnostics)))
print('Unresolved examples:',[d for d in diagnostics if d['diagnostic_status']=='unresolved'][:5])

Team-period combinations: 80
Status counts: {'consistent_provisional': 80}
Unresolved examples: []


In [4]:
csv_dest=OUT/'P2_attacking_direction_diagnostic.csv'
with csv_dest.open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=list(diagnostics[0]));writer.writeheader();writer.writerows(diagnostics)
report={
  'created_utc':datetime.now(timezone.utc).isoformat(),
  'purpose':'Provisional attacking direction diagnostic using goalkeeper position, not a validated axis convention',
  'n_matches':len(paths),
  'team_period_rows':len(diagnostics),
  'status_counts':dict(Counter(d['diagnostic_status'] for d in diagnostics)),
  'unresolved_rows':[d for d in diagnostics if d['diagnostic_status']=='unresolved'],
  'all_rows':diagnostics,
  'cautions':[
   'Goalkeeper median sign is a hypothesis, not verified attacking direction',
   'Only directly detected goalkeeper positions determine provisional signs',
   'Positional extrapolation and imputation never used to set direction',
   'Results must be corroborated with coordinate documentation or representative visuals',
   'Unresolved team-period cases get no signed progression feature',
   'Goalkeeper sampling is conditioned on possession-start frames, not full-match trajectories',
   'No hypothesis tests, predictive models or claims about pressing effectiveness']}
gate=OUT/'P2_attacking_direction_gate.json'
gate.write_text(json.dumps(report,indent=2))
print('Gate saved:',gate)
print('CSV saved:',csv_dest)

Gate saved: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P2_attacking_direction_gate.json
CSV saved: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P2_attacking_direction_diagnostic.csv


### Interpretation gate

This notebook only checks whether a direction hypothesis is **internally plausible**, not correct. Before building signed progression metrics, confirm the goalkeeper-side hypothesis using SkillCorner coordinate definitions or match visuals, including halves and teams. If any goalkeeper evidence is missing or opposing signs conflict, do **not** guess; inspect those team-periods specifically. Player/ball spatial geometry and unsiged distances remain valid without a direction assignment.